# Models & theory: practice notebook

2 exercises, Easy to Hard, each with hidden tests.

**How to use it**
1. Run the three setup cells below once (Runtime → Run all works too; the exercise cells just print `Wrong answer` until you solve them).
2. For each exercise, write your code in the solution cell, run it, then run the `check(...)` cell under it. It prints **Accepted** or which test failed and why.
3. Stuck? `solution("m9")` prints the reference solution for that exercise.

Opened from the roadmap it runs in Google Colab. To keep your work, use File → Save a copy in Drive the first time.

In [ ]:
# Install anything missing (Colab already has numpy, pandas, scipy, scikit-learn and matplotlib)
import importlib.util, subprocess, sys
for _p in []:
    if importlib.util.find_spec(_p) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p], check=False)


## Setup: data
The datasets every exercise uses. Read them if you like; don't change them.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
_g = np.random.default_rng(2026)
X = _g.normal(size=(300, 3))
y = 1.5 + X @ np.array([2.0, -1.0, 0.5]) + _g.normal(0, 1, 300)
XL = _g.normal(size=(200, 10))
XL[:, 1] = 0.8 * XL[:, 0] + 0.6 * XL[:, 1]
yL = 2.0 + XL @ np.array([3, -2, 0, 0, 1.5, 0, 0, 0, 0, 0.0]) + _g.normal(0, 1, 200)
XC = _g.normal(size=(600, 3))
yC = (_g.uniform(size=600) < 1 / (1 + np.exp(-(-0.5 + XC @ np.array([1.2, -0.8, 0.3]))))).astype(int)
_f = _g.normal(size=(400, 2))
Z = _f @ _g.normal(size=(2, 6)) + 0.3 * _g.normal(size=(400, 6))
_T = ["AAA", "BBB", "CCC", "DDD"]
_cov = 0.0001 * np.array([[4, 2, 1, 0.5], [2, 3, 1, 0.5], [1, 1, 2, 0.2], [0.5, 0.5, 0.2, 1]])
_r = _g.multivariate_normal([0.0006, 0.0004, 0.0003, 0.0001], _cov, 500)
_r[_g.uniform(size=_r.shape) < 0.01] = np.nan
rets = pd.DataFrame(_r, index=pd.bdate_range("2024-01-01", periods=500), columns=_T)
_m = 2000
credit = pd.DataFrame({
    "income": np.round(_g.lognormal(11, 0.4, _m), -2),
    "debt_ratio": np.round(_g.beta(2, 5, _m), 3),
    "age": _g.integers(21, 70, _m),
    "late_payments": _g.poisson(0.6, _m),
})
_z = -1.0 - 0.00002 * (credit.income - 60000) + 3.0 * credit.debt_ratio - 0.02 * (credit.age - 40) + 0.7 * credit.late_payments
credit["default"] = (_g.uniform(size=_m) < 1 / (1 + np.exp(-_z))).astype(int)
mu = np.array([0.06, 0.08, 0.10, 0.04, 0.12])
_s = np.array([0.15, 0.20, 0.25, 0.08, 0.30])
_rho = np.array([[1, .5, .3, .1, .4], [.5, 1, .4, .1, .5], [.3, .4, 1, 0, .6], [.1, .1, 0, 1, 0], [.4, .5, .6, 0, 1]])
Sigma = np.outer(_s, _s) * _rho
_lv = np.cumsum(_g.normal(0, 0.3, 200)) + 10
level_obs = _lv + _g.normal(0, 1.0, 200)
level_obs[[40, 41, 42, 120]] = np.nan
hx = 50 + np.cumsum(_g.normal(0, 1, 300))
_beta = 1.2 + np.cumsum(_g.normal(0, 0.01, 300))
hy = _beta * hx + _g.normal(0, 1.5, 300)
del _g

In [ ]:
# Test runner: check(id) and solution(id)
import base64 as _b64, json as _json, io as _io, contextlib as _ctx, traceback as _tb
_LAB = _json.loads(_b64.b64decode("eyJtOSI6IHsic29sdXRpb24iOiAiaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdFxuXG5kZWYgc3VtbWFyeShyZXRzKTpcbiAgICB3ZWFsdGggPSAoMSArIHJldHMuZmlsbG5hKDApKS5jdW1wcm9kKClcbiAgICBvdXQgPSBwZC5EYXRhRnJhbWUoe1xuICAgICAgICBcImFubl9yZXR1cm5cIjogcmV0cy5tZWFuKCkgKiAyNTIsXG4gICAgICAgIFwiYW5uX3ZvbFwiOiByZXRzLnN0ZCgpICogbnAuc3FydCgyNTIpLFxuICAgIH0pXG4gICAgb3V0W1wic2hhcnBlXCJdID0gb3V0LmFubl9yZXR1cm4gLyBvdXQuYW5uX3ZvbFxuICAgIG91dFtcIm1heF9kcmF3ZG93blwiXSA9ICh3ZWFsdGggLyB3ZWFsdGguY3VtbWF4KCkgLSAxKS5taW4oKVxuICAgIHJldHVybiBvdXQuc29ydF92YWx1ZXMoXCJzaGFycGVcIiwgYXNjZW5kaW5nPUZhbHNlKVxuXG5kZWYgcGxvdChyZXRzKTpcbiAgICBmaWcsIChheDEsIGF4MikgPSBwbHQuc3VicGxvdHMoMSwgMiwgZmlnc2l6ZT0oMTEsIDQpKVxuICAgICgxICsgcmV0cy5maWxsbmEoMCkpLmN1bXByb2QoKS5wbG90KGF4PWF4MSlcbiAgICBheDEuc2V0X3RpdGxlKFwiQ3VtdWxhdGl2ZSB3ZWFsdGhcIik7IGF4MS5zZXRfeWxhYmVsKFwiZ3Jvd3RoIG9mIDFcIilcbiAgICBjID0gcmV0cy5jb3JyKClcbiAgICBpbSA9IGF4Mi5pbXNob3coYywgdm1pbj0tMSwgdm1heD0xLCBjbWFwPVwiUmRCdV9yXCIpXG4gICAgYXgyLnNldF94dGlja3MocmFuZ2UobGVuKGMpKSwgYy5jb2x1bW5zKTsgYXgyLnNldF95dGlja3MocmFuZ2UobGVuKGMpKSwgYy5jb2x1bW5zKVxuICAgIGF4Mi5zZXRfdGl0bGUoXCJDb3JyZWxhdGlvblwiKTsgZmlnLmNvbG9yYmFyKGltLCBheD1heDIpXG4gICAgZmlnLnRpZ2h0X2xheW91dCgpXG4iLCAidGVzdHMiOiAiXG5mcm9tIHBhbmRhcy50ZXN0aW5nIGltcG9ydCBhc3NlcnRfZnJhbWVfZXF1YWwgYXMgX2FmZVxuZGVmIF90KGRmKTpcbiAgICBhID0gVVtcInN1bW1hcnlcIl0oZGYpOyBhc3NlcnQgYSBpcyBub3QgTm9uZSwgXCJzdW1tYXJ5IHJldHVybmVkIE5vbmVcIlxuICAgIGFzc2VydCBpc2luc3RhbmNlKGEsIHBkLkRhdGFGcmFtZSksIGZcImV4cGVjdGVkIGEgRGF0YUZyYW1lLCBnb3Qge3R5cGUoYSkuX19uYW1lX199XCJcbiAgICBiID0gUltcInN1bW1hcnlcIl0oZGYpXG4gICAgYXNzZXJ0IGxpc3QoYS5jb2x1bW5zKSA9PSBsaXN0KGIuY29sdW1ucyksIGZcImNvbHVtbnMge2xpc3QoYS5jb2x1bW5zKX0sIGV4cGVjdGVkIHtsaXN0KGIuY29sdW1ucyl9XCJcbiAgICBhc3NlcnQgbGlzdChhLmluZGV4KSA9PSBsaXN0KGIuaW5kZXgpLCBmXCJyb3cgb3JkZXIge2xpc3QoYS5pbmRleCl9LCBleHBlY3RlZCB7bGlzdChiLmluZGV4KX1cIlxuICAgIF9hZmUoYSwgYiwgY2hlY2tfZXhhY3Q9RmFsc2UsIHJ0b2w9MWUtNiwgY2hlY2tfbmFtZXM9RmFsc2UsIGNoZWNrX2R0eXBlPUZhbHNlKVxuVEVTVFMgPSBbKFwiZnVsbCBzYW1wbGVcIiwgbGFtYmRhOiBfdChyZXRzKSksIChcImhpZGRlbjogMjAyNCBvbmx5XCIsIGxhbWJkYTogX3QocmV0cy5sb2NbXCIyMDI0XCJdKSldIiwgImxhbmciOiAicHkifSwgIm0xMCI6IHsic29sdXRpb24iOiAiZnJvbSBza2xlYXJuLm1vZGVsX3NlbGVjdGlvbiBpbXBvcnQgdHJhaW5fdGVzdF9zcGxpdFxuZnJvbSBza2xlYXJuLnBpcGVsaW5lIGltcG9ydCBQaXBlbGluZVxuZnJvbSBza2xlYXJuLnByZXByb2Nlc3NpbmcgaW1wb3J0IFN0YW5kYXJkU2NhbGVyXG5mcm9tIHNrbGVhcm4ubGluZWFyX21vZGVsIGltcG9ydCBMb2dpc3RpY1JlZ3Jlc3Npb25cbmZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCByb2NfYXVjX3Njb3JlXG5cbkZFQVRVUkVTID0gW1wiaW5jb21lXCIsIFwiZGVidF9yYXRpb1wiLCBcImFnZVwiLCBcImxhdGVfcGF5bWVudHNcIl1cblxuZGVmIGZpdF9kZWZhdWx0KGRmKTpcbiAgICBYLCB5ID0gZGZbRkVBVFVSRVNdLCBkZltcImRlZmF1bHRcIl1cbiAgICBYdHIsIFh0ZSwgeXRyLCB5dGUgPSB0cmFpbl90ZXN0X3NwbGl0KFgsIHksIHRlc3Rfc2l6ZT0wLjMsIHJhbmRvbV9zdGF0ZT0wLCBzdHJhdGlmeT15KVxuICAgIG1vZGVsID0gUGlwZWxpbmUoWyhcInNjYWxlXCIsIFN0YW5kYXJkU2NhbGVyKCkpLCAoXCJsclwiLCBMb2dpc3RpY1JlZ3Jlc3Npb24oKSldKS5maXQoWHRyLCB5dHIpXG4gICAgYXVjID0gcm9jX2F1Y19zY29yZSh5dGUsIG1vZGVsLnByZWRpY3RfcHJvYmEoWHRlKVs6LCAxXSlcbiAgICBjb2VmcyA9IG1vZGVsLm5hbWVkX3N0ZXBzW1wibHJcIl0uY29lZl9bMF1cbiAgICByZXR1cm4ge1wiYXVjXCI6IGZsb2F0KGF1YyksIFwib2Rkc19yYXRpb3NcIjoge2Y6IGZsb2F0KG5wLmV4cChjKSkgZm9yIGYsIGMgaW4gemlwKEZFQVRVUkVTLCBjb2Vmcyl9fVxuIiwgInRlc3RzIjogIlxuZGVmIGNsb3NlKGEsIGIsIHJlbD0xZS02LCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhID0gZmxvYXQoYSk7IGIgPSBmbG9hdChiKVxuICAgIGFzc2VydCBhYnMoYSAtIGIpIDw9IHJlbCAqIG1heCgxLjAsIGFicyhiKSksIGZcInt3aGF0fSA9IHthOi42Z30sIGV4cGVjdGVkIHtiOi42Z31cIlxuZGVmIGJldHdlZW4oYSwgbG8sIGhpLCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhc3NlcnQgbG8gPD0gZmxvYXQoYSkgPD0gaGksIGZcInt3aGF0fSA9IHtmbG9hdChhKTouNGZ9LCBleHBlY3RlZCBiZXR3ZWVuIHtsb30gYW5kIHtoaX1cIlxuXG5kZWYgX3QoZGYpOlxuICAgIGEgPSBVW1wiZml0X2RlZmF1bHRcIl0oZGYpOyBhc3NlcnQgaXNpbnN0YW5jZShhLCBkaWN0KSwgXCJyZXR1cm4gYSBkaWN0IHdpdGggJ2F1YycgYW5kICdvZGRzX3JhdGlvcydcIlxuICAgIGIgPSBSW1wiZml0X2RlZmF1bHRcIl0oZGYpXG4gICAgY2xvc2UoYS5nZXQoXCJhdWNcIiksIGJbXCJhdWNcIl0sIDFlLTYsIFwiYXVjXCIpXG4gICAgbyA9IGEuZ2V0KFwib2Rkc19yYXRpb3NcIikgb3Ige31cbiAgICBhc3NlcnQgc2V0KG8pID09IHNldChiW1wib2Rkc19yYXRpb3NcIl0pLCBmXCJvZGRzX3JhdGlvcyBrZXlzIHtzb3J0ZWQobyl9LCBleHBlY3RlZCB7c29ydGVkKGJbJ29kZHNfcmF0aW9zJ10pfVwiXG4gICAgZm9yIGYgaW4gYltcIm9kZHNfcmF0aW9zXCJdOlxuICAgICAgICBjbG9zZShvW2ZdLCBiW1wib2Rkc19yYXRpb3NcIl1bZl0sIDFlLTQsIGZcIm9kZHMgcmF0aW8gb2Yge2Z9XCIpXG5URVNUUyA9IFsoXCJmdWxsIGRhdGFcIiwgbGFtYmRhOiBfdChjcmVkaXQpKSwgKFwiaGlkZGVuOiBmaXJzdCAxLDIwMCByb3dzXCIsIGxhbWJkYTogX3QoY3JlZGl0Lmlsb2NbOjEyMDBdKSldIiwgImxhbmciOiAicHkifX0=").decode())
_SETUP = _b64.b64decode("CmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmZyb20gc2NpcHkgaW1wb3J0IHN0YXRzCl9nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKDIwMjYpClggPSBfZy5ub3JtYWwoc2l6ZT0oMzAwLCAzKSkKeSA9IDEuNSArIFggQCBucC5hcnJheShbMi4wLCAtMS4wLCAwLjVdKSArIF9nLm5vcm1hbCgwLCAxLCAzMDApClhMID0gX2cubm9ybWFsKHNpemU9KDIwMCwgMTApKQpYTFs6LCAxXSA9IDAuOCAqIFhMWzosIDBdICsgMC42ICogWExbOiwgMV0KeUwgPSAyLjAgKyBYTCBAIG5wLmFycmF5KFszLCAtMiwgMCwgMCwgMS41LCAwLCAwLCAwLCAwLCAwLjBdKSArIF9nLm5vcm1hbCgwLCAxLCAyMDApClhDID0gX2cubm9ybWFsKHNpemU9KDYwMCwgMykpCnlDID0gKF9nLnVuaWZvcm0oc2l6ZT02MDApIDwgMSAvICgxICsgbnAuZXhwKC0oLTAuNSArIFhDIEAgbnAuYXJyYXkoWzEuMiwgLTAuOCwgMC4zXSkpKSkpLmFzdHlwZShpbnQpCl9mID0gX2cubm9ybWFsKHNpemU9KDQwMCwgMikpClogPSBfZiBAIF9nLm5vcm1hbChzaXplPSgyLCA2KSkgKyAwLjMgKiBfZy5ub3JtYWwoc2l6ZT0oNDAwLCA2KSkKX1QgPSBbIkFBQSIsICJCQkIiLCAiQ0NDIiwgIkRERCJdCl9jb3YgPSAwLjAwMDEgKiBucC5hcnJheShbWzQsIDIsIDEsIDAuNV0sIFsyLCAzLCAxLCAwLjVdLCBbMSwgMSwgMiwgMC4yXSwgWzAuNSwgMC41LCAwLjIsIDFdXSkKX3IgPSBfZy5tdWx0aXZhcmlhdGVfbm9ybWFsKFswLjAwMDYsIDAuMDAwNCwgMC4wMDAzLCAwLjAwMDFdLCBfY292LCA1MDApCl9yW19nLnVuaWZvcm0oc2l6ZT1fci5zaGFwZSkgPCAwLjAxXSA9IG5wLm5hbgpyZXRzID0gcGQuRGF0YUZyYW1lKF9yLCBpbmRleD1wZC5iZGF0ZV9yYW5nZSgiMjAyNC0wMS0wMSIsIHBlcmlvZHM9NTAwKSwgY29sdW1ucz1fVCkKX20gPSAyMDAwCmNyZWRpdCA9IHBkLkRhdGFGcmFtZSh7CiAgICAiaW5jb21lIjogbnAucm91bmQoX2cubG9nbm9ybWFsKDExLCAwLjQsIF9tKSwgLTIpLAogICAgImRlYnRfcmF0aW8iOiBucC5yb3VuZChfZy5iZXRhKDIsIDUsIF9tKSwgMyksCiAgICAiYWdlIjogX2cuaW50ZWdlcnMoMjEsIDcwLCBfbSksCiAgICAibGF0ZV9wYXltZW50cyI6IF9nLnBvaXNzb24oMC42LCBfbSksCn0pCl96ID0gLTEuMCAtIDAuMDAwMDIgKiAoY3JlZGl0LmluY29tZSAtIDYwMDAwKSArIDMuMCAqIGNyZWRpdC5kZWJ0X3JhdGlvIC0gMC4wMiAqIChjcmVkaXQuYWdlIC0gNDApICsgMC43ICogY3JlZGl0LmxhdGVfcGF5bWVudHMKY3JlZGl0WyJkZWZhdWx0Il0gPSAoX2cudW5pZm9ybShzaXplPV9tKSA8IDEgLyAoMSArIG5wLmV4cCgtX3opKSkuYXN0eXBlKGludCkKbXUgPSBucC5hcnJheShbMC4wNiwgMC4wOCwgMC4xMCwgMC4wNCwgMC4xMl0pCl9zID0gbnAuYXJyYXkoWzAuMTUsIDAuMjAsIDAuMjUsIDAuMDgsIDAuMzBdKQpfcmhvID0gbnAuYXJyYXkoW1sxLCAuNSwgLjMsIC4xLCAuNF0sIFsuNSwgMSwgLjQsIC4xLCAuNV0sIFsuMywgLjQsIDEsIDAsIC42XSwgWy4xLCAuMSwgMCwgMSwgMF0sIFsuNCwgLjUsIC42LCAwLCAxXV0pClNpZ21hID0gbnAub3V0ZXIoX3MsIF9zKSAqIF9yaG8KX2x2ID0gbnAuY3Vtc3VtKF9nLm5vcm1hbCgwLCAwLjMsIDIwMCkpICsgMTAKbGV2ZWxfb2JzID0gX2x2ICsgX2cubm9ybWFsKDAsIDEuMCwgMjAwKQpsZXZlbF9vYnNbWzQwLCA0MSwgNDIsIDEyMF1dID0gbnAubmFuCmh4ID0gNTAgKyBucC5jdW1zdW0oX2cubm9ybWFsKDAsIDEsIDMwMCkpCl9iZXRhID0gMS4yICsgbnAuY3Vtc3VtKF9nLm5vcm1hbCgwLCAwLjAxLCAzMDApKQpoeSA9IF9iZXRhICogaHggKyBfZy5ub3JtYWwoMCwgMS41LCAzMDApCmRlbCBfZwo=").decode()

def _err(e):
    fr = [f for f in _tb.extract_tb(e.__traceback__) if f.filename.startswith("<ipython") or f.filename == "<string>"]
    return f"{type(e).__name__}: {e}"

def check(ex_id, query=None):
    """Run the hidden tests for one exercise, like Submit in the editor."""
    e = _LAB[ex_id]
    ref = {}
    with _ctx.redirect_stdout(_io.StringIO()):
        exec(_SETUP, ref)
        if e["lang"] == "sql":
            ref["__sql__"] = e["solution"]
        else:
            exec(e["solution"], ref)
    if e["lang"] == "sql":
        if query is None:
            print("Pass your query string: check(\"" + ex_id + "\", q)")
            return
        user = {"__sql__": query}
    else:
        user = dict(globals())
    tns = dict(ref)
    tns["U"], tns["R"] = user, ref
    exec(e["tests"], tns)
    ok, tests = 0, tns["TESTS"]
    for name, fn in tests:
        try:
            fn()
            print("  ✓ " + name)
            ok += 1
        except AssertionError as err:
            print("  ✗ " + name + ": " + (str(err) or "wrong answer"))
        except Exception as err:
            print("  ✗ " + name + ": " + _err(err))
    print(("Accepted" if ok == len(tests) else "Wrong answer") + f" · {ok} / {len(tests)} test cases passed")

def solution(ex_id):
    """Print the reference solution. Try once more first."""
    print(_LAB[ex_id]["solution"])

---
## 1. Explore returns with pandas and matplotlib
*Medium · id `m9`*

<p><code>rets</code> is a pandas DataFrame of daily returns for four tickers (dates as index), with a few missing values.</p>
<p>1. Write <code>summary(rets)</code> returning a DataFrame indexed by ticker with columns <code>ann_return</code> (mean × 252), <code>ann_vol</code> (std × √252), <code>sharpe</code> (their ratio) and <code>max_drawdown</code> (the most negative wealth / running peak − 1, with wealth = cumulative product of 1 + r, <b>treating missing returns as 0</b> for this column only). Sort by <code>sharpe</code>, highest first.</p>
<p>2. Write <code>plot(rets)</code> drawing cumulative wealth for each ticker and a correlation heatmap. Run shows your figures; plots aren't graded.</p>

**Theory to know cold**
<ul><li>The first five minutes of any data task: shape, dtypes, <code>describe()</code>, missing values, duplicates, the date range, outliers. Say out loud what you check and why.</li>
<li>pandas vs Polars: index-based vs expression-based, eager vs lazy, NaN vs null. Be able to do this problem in both.</li>
<li>matplotlib: <code>fig, ax = plt.subplots()</code>, label axes, add a title and legend, <code>imshow</code> or <code>matshow</code> for a heatmap with a colourbar.</li></ul>

In [ ]:
import matplotlib.pyplot as plt

def summary(rets: pd.DataFrame) -> pd.DataFrame:
    # your code here
    pass

def plot(rets: pd.DataFrame):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    # your code here

# Sample run
try:
    print(rets.describe().round(5))
    print(summary(rets))
    plot(rets)
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("m9")

---
## 2. Fit and interpret a scikit-learn model
*Medium · id `m10`*

<p><code>credit</code> has <code>income, debt_ratio, age, late_payments</code> and a 0/1 <code>default</code> column.</p>
<p>Write <code>fit_default(df)</code>: split with <code>train_test_split(test_size=0.3, random_state=0, stratify=y)</code>, fit a <code>Pipeline</code> of <code>StandardScaler</code> then <code>LogisticRegression()</code> (defaults) on the training set, and return a dict with <code>"auc"</code> (ROC AUC on the test set) and <code>"odds_ratios"</code> (a dict of feature → exp(coefficient)).</p>
<p>Then explain each odds ratio in one sentence, as you would to an interviewer.</p>

**Theory to know cold**
<ul><li>With scaled inputs, an odds ratio is the multiplicative change in the odds of default per one standard deviation of that feature, holding the others fixed.</li>
<li><code>LogisticRegression</code> is ℓ2-regularised by default (C = 1), so coefficients are shrunk; scikit-learn gives no p-values. For inference use statsmodels <code>Logit</code>.</li>
<li>AUC = probability a random defaulter scores above a random non-defaulter. It ignores calibration; check a reliability curve or the Brier score too.</li>
<li>Fit the scaler inside the pipeline so the test set doesn't leak into it. Stratify when classes are imbalanced. Use cross-validation for model choice, the test set once.</li></ul>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

FEATURES = ["income", "debt_ratio", "age", "late_payments"]

def fit_default(df):
    # your code here
    pass

# Sample run
try:
    print(credit.describe().round(2))
    print(fit_default(credit))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("m10")